# Day 6 ML/RL Prep

5 basic mixed questions.

Instructions:
- Answer with reasoning, not just the final result.
- Keep answers short and precise.
- If you use code, explain what the tensor means.


## Question 1: `unsqueeze` for broadcasting

You have:
- `mask`: shape `(B, T) = (2, 4)`
- `x`: shape `(B, T, D) = (2, 4, 3)`

You want `mask` to multiply `x` across the feature dimension.

What shape should `mask` have after `unsqueeze`?
Which dimension would you unsqueeze?
Why does that make broadcasting work?


In [9]:
# Question 1: Answer and reasoning
import torch

mask = torch.randn(2, 4)
x = torch.randn(2, 4, 3)
mask.unsqueeze(-1).shape

# Broadcasting works from the right most to left most dimention. 
# It will be successfull if the dimentions match, or one of them is a 1. As the value in that dimention will be expanded.
# However, the number of dimentions in each tensor have to match x has an extra 'D' dimention that the mask does not have. 
# The solution here is to unsqueeze the mask along the third dimention with either mask.unsqueeze(-1) or mask.unsqueeze(2).
# This creates the dimention of size 1, which allows the masking to take place for the previously mentioned reason. 

torch.Size([2, 4, 1])

## Question 2: `gather` shape intuition

You have:
- `logits`: shape `(B, A) = (3, 5)`
- `actions`: shape `(B,) = (3,)`

You want the score for the chosen action in each row.

If you use `gather`, what shape should the index tensor have?
What shape will the gathered result have before and after a squeeze?


In [28]:
# Question 2: Answer and reasoning

logits = torch.randn(3, 5)
actions = torch.randint(0, 5, (3,))
scores = torch.gather(logits, 1, actions.unsqueeze(-1)).squeeze(-1)

# If you use gather, the index tensor will be actions, 
# But gather requires the input tensor (logits) to be the same shape as the index tensor (actions).
# To make both tensors the same shape, allowing gather to compute, actions needs an extra trailing dimention, making it (B,T,1).
# After the squeeze on the resulting scores tensor the trailing dimention is removed and has the shape (3,) or (B,)

## Question 3: Per-step vs per-trajectory returns

Consider:

```python
logits = policy(obs)          # (B, T, A)
dist = torch.distributions.Categorical(logits=logits)
logp = dist.log_prob(actions) # (B, T)
loss = -(logp * returns).mean()
```

What is the difference between:
- `returns.shape == (B,)`
- `returns.shape == (B, T)`

Which one matches a per-time-step loss directly, and why?


In [34]:
# Question 3: Answer and reasoning

b, t, a = 2, 3, 4

logits = torch.randn(b, t, a)
actions = torch.randint(0, a, (b, t))
returns = torch.randn(b, t)

dist = torch.distributions.Categorical(logits=logits)
logp = dist.log_prob(actions)
loss = -(logp * returns).mean()
loss

# If returns had the shape (B,) that would be the total return for the entire trajectory. 
# Whereas if returns had shape (B,T) then the return would be split into the reward for each timestep in the trajectory. 
# If you want per-time-step loss directly, you would require the per-time-step reward, in returns.shape = (B, T)

tensor(-1.1856)

## Question 4: Debugging a bad flatten

You have:
- `logits`: shape `(B, T, A) = (2, 3, 4)`
- `actions`: shape `(B, T) = (2, 3)`

Someone writes:

```python
logits_flat = logits.reshape(-1)
actions_flat = actions.reshape(-1)
loss = F.cross_entropy(logits_flat, actions_flat)
```

What is wrong with `logits_flat` here?
What shape should it have instead?


In [50]:
# Question 4: Answer and reasoning

b,t,a = 2,3,4
logits = torch.randn(b, t, a)
actions = torch.randint(0, a, (b, t))

logits_flat = logits.reshape(-1, a)
actions_flat = actions.reshape(-1)
logits_flat.shape, actions_flat.shape

loss = torch.nn.functional.cross_entropy(logits_flat, actions_flat)
loss

# Logits flat here would have shape (24,)
# Cross entropy requires the logits to be shape (N, C), Samples and classes.
# The classes in this example is the actions dimention, so this must be maintained.
# The correct reshape for logits would be reshape(-1,a), resulting in logit_flat being shape (b*t, a).

tensor(1.4827)

## Question 5: Small implementation idea

You have:
- `x`: shape `(B, T, D)`
- `mask`: shape `(B, T)`

You want to zero out invalid time steps in `x`.

Do not write full polished code unless you want to.
I want:
- what shape `mask` needs before multiplication
- how you would get that shape
- what the result shape will be


In [57]:
# Question 5: Answer and reasoning

b, t, a, = 2, 4, 5
x = torch.randn(b, t, a)
mask = torch.randn(b, t).unsqueeze(-1)
(x*mask).shape

# Mask(B,T)*x(B,T,A) cannot be computed as the process will attempt to match dimentions from left to right. 
# With success condition being if they match or one dimention is 1. The value at this dimention gets expanded accross D. 
# For mask*x to be computed a trailing dimention must be added to mask, giving it the shape (b,t,1).
# The resulting tensor of x*mask will be shape (B,T,A).

torch.Size([2, 4, 5])